In [9]:
import gc
gc.collect()

464

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import register_and_resample, sikit_to_just_data
from core.helper import load_NiFty_and_save_raw_data , copy_NiFty, patients_transform
from core.stat_calc import find_periods
from core.visualization_lib import show_transformation

from core.transformers.nifti_to_raw_transformer import nifti_to_raw_transformer


In [11]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = {
    'nifi_to_raw' : {'start': '0_nifty', 'end': '1_raw'},
    'filling_anatomy_gaps' : {'start': '1_raw', 'end': '2_anatomy_gap_filled'}
}

crop_size = (160,160,24)

In [12]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

copy data to preprocess folder

In [13]:
copy_NiFty(orginal_data_folder, file_manager,channels, filter=['3322'], step=preprocessed_steps['nifi_to_raw']['start'],)

transform to raw

In [14]:
start_step = preprocessed_steps['nifi_to_raw']['start']
end_step = preprocessed_steps['nifi_to_raw']['end']

to_raw_transform = nifti_to_raw_transformer()

patients_transform(file_manager, start_step, end_step, to_raw_transform)

dispaly

In [ ]:
folder_shower(file_manager,'1_raw',normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…

remember about allingning

<h2>Working with holes in prostate layer</h2>

In [16]:
from core.anatomy_gap_fixer import find_gaps_in_anatomy
from core.anatomy_gap_fixer import fix_patient_anatomy

checking out outliers

In [ ]:
patinets = file_manager.get_file_names()
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
for outlier in outliers:
    print(outlier)
    prostate = file_manager.load_file('raw',outlier)['anatomy'] 
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    true_indices = np.where(valid_layers)[0]
    periods = find_periods(true_indices)
    print(len(periods))
    print(periods)

NameError: name 'patients' is not defined

FIX outliers

In [ ]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
print(outliers)
start_step = 'raw'
end_step = 'raw'

for outlier in outliers:
    outlier_data = file_manager.load_file(start_step, outlier)
    outlier_new_data = fix_patient_anatomy(outlier_data)
    file_manager.save_file(end_step,outlier,outlier_new_data)

['3322']
20  22


<h2>CROPPING!!!</h2>

Cutting pictures into correct sizes

two ways of centering

finding maximum prostate dimentions

In [ ]:
from core.stat_calc import find_patients_max_prostate_sizes

step = 'raw'
maximum_prostate_size = find_patients_max_prostate_sizes(patients,file_manager)
print(maximum_prostate_size)

[67, 63, 16]


center cropping

In [ ]:
from core.cropping_lib import center_crop_shift
from core.stat_calc import find_centroid_non_weighted

THERE IS NO PADDING!!!!

In [ ]:
step = 'raw'

for patient in patients:

    data = file_manager.load_file(step, patient)
    center = find_centroid_non_weighted(data['anatomy'])
    for channel in data:
        data[channel] = center_crop_shift(data[channel],center,crop_size)    
    file_manager.save_file('cropped',patient,data)
    

In [ ]:
folder_shower(file_manager,normalizer=normalize_volume)
folder_shower(file_manager,step = 'cropped',normalizer=normalize_volume)

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…

interactive(children=(Dropdown(description='Patient:', options=('3322',), value='3322'), Dropdown(description=…